# PFAS Groundwater — T1 Binary Baseline (Colab)

**Task T1a**: predict whether a well exceeds the regulatory PFAS threshold (binary).

This notebook is **AUTONOMOUS** (CLAUDE.md §4): it bootstraps `src/` and the dataset via
`git clone` (no Google Drive required), then runs the full baseline pipeline
(LR / RF / XGBoost, spatial + random CV, SHAP, ablations).

**Strict predictive mode**: no PFAS measurement is used as a feature (CLAUDE.md §1).

---
**PARAMETERS TO SET (cell below)**:
- `SMOKE_TEST` : `True` for a fast CPU sanity check (~3 min), `False` for the full GPU run.
- `REPO_URL` : GitHub repo URL (default `https://github.com/dnwiloic/pfas-gnn.git`).
- `GIT_REF` : branch or full commit SHA to pin (default `main`).
- `DATA_PATH` : path to the parquet file, relative to the cloned repo root
  (default `data/CA-PFAS-ASGWS.parquet` — the file is versioned in the repo).
- `TARGET` : `"T1a"` (default) or `"T1b"`.

In [1]:
# ============================================================
#  USER PARAMETERS — edit these before running
# ============================================================

SMOKE_TEST = False          # True = fast CPU sanity check; False = full GPU run

# --- Repository (code + data) ---
# The dataset is versioned in the repo at data/CA-PFAS-ASGWS.parquet.
# git clone brings both src/ and data/ automatically. No Drive needed.
REPO_URL = "https://github.com/dnwiloic/pfas-gnn.git"   # <-- SET THIS if forked
GIT_REF  = "main"          # branch name or full commit SHA

# Path to the parquet file RELATIVE to the cloned repo root.
# Change only if the file was moved inside the repo.
DATA_PATH = "data/CA-PFAS-ASGWS.parquet"

# --- Run target ---
TARGET = "T1a"   # "T1a" (EPA MCL regulatory) or "T1b" (sum > 70 ng/L)

print(f"SMOKE_TEST={SMOKE_TEST}  REPO_URL={REPO_URL!r}  GIT_REF={GIT_REF!r}  TARGET={TARGET}")

SMOKE_TEST=False  REPO_URL='https://github.com/dnwiloic/pfas-gnn.git'  GIT_REF='main'  TARGET=T1a


## Cell 1 — GPU detection & version info

In [2]:
import sys, platform
print(f"Python  : {sys.version}")
print(f"Platform: {platform.platform()}")

# Detect Colab environment
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass
print(f"IN_COLAB: {IN_COLAB}")

if IN_COLAB:
    import shutil, subprocess
    if shutil.which("nvidia-smi") is None:        # CPU runtime: binary absent
        class _R: returncode = 1; stdout = ""
        r = _R()
    else:
        r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if r.returncode == 0:
        print("\n[GPU detected]")
        print(r.stdout[:800])
    else:
        print("[WARNING] No GPU found — set Runtime > Change runtime type > GPU in Colab.")
        print("Note: scikit-learn and XGBoost use CPU cores (n_jobs=-1).")
        print("      A High-RAM CPU instance may be efficient for this baseline.")
else:
    print("[Local run — GPU check skipped]")

# Library versions
for pkg_name in ["numpy", "sklearn", "xgboost", "pandas"]:
    try:
        mod = __import__(pkg_name)
        ver = getattr(mod, "__version__", "?")
        print(f"{pkg_name:12s}: {ver}")
    except ImportError:
        print(f"{pkg_name:12s}: NOT INSTALLED")

Python  : 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
Platform: Linux-6.17.0-35-generic-x86_64-with-glibc2.39
IN_COLAB: False
[Local run — GPU check skipped]
numpy       : 1.26.4
sklearn     : 1.8.0
xgboost     : 2.0.3
pandas      : 2.1.4


## Cell 2 — Install dependencies (Colab only)

In [3]:
if IN_COLAB:
    import subprocess, sys
    pkgs = [
        "xgboost>=2.0,<3.0",
        "optuna>=3.5,<4.0",
        "shap>=0.44,<1.0",
        "imbalanced-learn>=0.11,<1.0",
        "pyyaml>=6.0,<7.0",
        "pandas>=2.0,<3.0",
        "scikit-learn>=1.4,<2.0",
        "pyarrow>=14.0",
        "scipy>=1.11",
    ]
    print("Installing packages...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("[ERROR] pip install failed:")
        print(result.stderr[-1000:])
        raise RuntimeError("Dependency installation failed.")
    print("Packages installed. Verifying imports...")
    import xgboost, optuna, shap, imblearn, yaml, pandas, sklearn
    print(f"  xgboost={xgboost.__version__}  optuna={optuna.__version__}  "
          f"shap={shap.__version__}  imblearn={imblearn.__version__}")
    print("Imports OK.")
else:
    print("[Local run] Skipping pip install — using local environment.")

[Local run] Skipping pip install — using local environment.


## Cell 3 — Bootstrap `src/` and dataset via `git clone`

The repo contains both `src/` (code) and `data/CA-PFAS-ASGWS.parquet` (dataset).
A single `git clone` brings both. **No Google Drive needed.**

**IMPORTANT**: `REPO_URL` and `GIT_REF` must point to an up-to-date remote.
If you have uncommitted local changes, push them first:
```bash
git push origin main
```

In [4]:
import sys, subprocess
from pathlib import Path

if IN_COLAB:
    REPO_LOCAL = Path("/content/pfas-gnn")
    if REPO_LOCAL.exists():
        print(f"Repo already cloned at {REPO_LOCAL} — updating to {GIT_REF} ...")
        subprocess.run(["git", "-C", str(REPO_LOCAL), "fetch", "origin"],
                       capture_output=True, text=True)
        subprocess.run(["git", "-C", str(REPO_LOCAL), "checkout", GIT_REF],
                       capture_output=True, text=True)
        r = subprocess.run(
            ["git", "-C", str(REPO_LOCAL), "reset", "--hard", f"origin/{GIT_REF}"],
            capture_output=True, text=True
        )
        print(r.stdout.strip() or r.stderr.strip())
    else:
        print(f"Cloning {REPO_URL} @ {GIT_REF} ...")
        r = subprocess.run(
            ["git", "clone", "--depth=1", "--branch", GIT_REF, REPO_URL, str(REPO_LOCAL)],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print("[ERROR] git clone failed:")
            print(r.stderr)
            raise RuntimeError(
                f"git clone failed. Check REPO_URL={REPO_URL!r} and GIT_REF={GIT_REF!r}.\n"
                "Ensure the branch exists and the repository is public (or provide auth)."
            )
        print("Cloned OK.")
else:
    # Local run: repo root is parent of notebooks/
    REPO_LOCAL = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path("..").resolve()
    print(f"[Local run] Using repo at {REPO_LOCAL}")

# Add repo root to sys.path so `from src.xxx import yyy` works
repo_str = str(REPO_LOCAL)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
print(f"sys.path[0] = {sys.path[0]}")

[Local run] Using repo at /home/wiloic/M2/Recherche/tmp/pfas/pfas-gnn
sys.path[0] = /home/wiloic/M2/Recherche/tmp/pfas/pfas-gnn


In [5]:
# --- Guard-rail: verify the code is present and up-to-date ---
import sys

# Force reimport in case src was already imported with an old path
for mod in list(sys.modules.keys()):
    if mod.startswith("src"):
        del sys.modules[mod]

try:
    import src.baselines_t1
    import src.baselines_t2
    import src.metrics
except ImportError as e:
    raise ImportError(
        f"CRITICAL: Cannot import src modules: {e}\n"
        f"The clone at {REPO_LOCAL} does not contain src/.\n"
        f"Check REPO_URL={REPO_URL!r} and GIT_REF={GIT_REF!r}."
    )

# Confirm FrequencyClassChain is present (proves code is not obsolete)
assert hasattr(src.baselines_t2, "FrequencyClassChain"), (
    "CRITICAL: FrequencyClassChain not found in src.baselines_t2.\n"
    f"The code at {REPO_LOCAL} is OBSOLETE.\n"
    f"Push the latest src/ to the remote (GIT_REF={GIT_REF!r}) then re-run this cell."
)

# Confirm the 5 required metrics are declared
from src.metrics import REQUIRED
assert set(REQUIRED) == {"roc_auc", "f1", "accuracy", "recall", "precision"}, (
    f"CRITICAL: src.metrics.REQUIRED={REQUIRED} — expected the 5 standard metrics. "
    "Code may be obsolete."
)

print("[guard-rail] src/ is up-to-date: FrequencyClassChain found, 5 metrics OK.")
from src.baselines_t1 import run_baselines
print("[guard-rail] run_baselines imported successfully.")

/home/wiloic/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[guard-rail] src/ is up-to-date: FrequencyClassChain found, 5 metrics OK.
[guard-rail] run_baselines imported successfully.


## GPU utilisation check

Confirms whether the GPU is actually exercised. **Only XGBoost uses the GPU** (`device='cuda'`); scikit-learn RandomForest / HistGradientBoosting / LogisticRegression are CPU-only. If no GPU is detected, tree models fall back to CPU automatically.

In [ ]:
# --- GPU utilisation check (XGBoost device='cuda') ---
from src import config as _C
print("GPU available :", _C.gpu_available())
print("XGBoost params:", _C.xgb_device_params())
if _C.gpu_available():
    import xgboost as _xgb, numpy as _np, time as _t
    _X = _np.random.rand(60000, 30); _y = (_np.random.rand(60000) < 0.3).astype(int)
    _t0 = _t.time()
    _xgb.XGBClassifier(n_estimators=80, **_C.xgb_device_params()).fit(_X, _y)
    print(f"XGBoost GPU fit OK in {_t.time()-_t0:.1f}s -> the GPU is being exercised.")
    print("Note: RandomForest / HistGradientBoosting / LogisticRegression stay on CPU (sklearn).")
else:
    print("No GPU detected -> tree models run on CPU.")
    print("For GPU: Runtime > Change runtime type > GPU (XGBoost will then use device='cuda').")


## Cell 4 — Load dataset from cloned repo

The dataset `data/CA-PFAS-ASGWS.parquet` is versioned in the repository and was
downloaded automatically by `git clone` in Cell 3. No Drive mount needed.

Integrity check: shape must be 46338 x 201 with keys `gm_well_id`, `latitude`,
`longitude`, `PFOA_ngL`. Any failure stops execution with an explicit message.

In [6]:
import pandas as pd
from pathlib import Path

# Resolve DATA_PATH relative to the cloned repo
data_file = REPO_LOCAL / DATA_PATH
if not data_file.exists():
    raise FileNotFoundError(
        f"Dataset not found at {data_file}.\n"
        f"Expected path: REPO_LOCAL/DATA_PATH = {REPO_LOCAL}/{DATA_PATH}\n"
        "The parquet file should be versioned in the repo. If missing:\n"
        "  1. Check that the file is committed: git ls-files data/\n"
        "  2. Or update DATA_PATH to its actual location within the repo."
    )

print(f"Dataset: {data_file}  size={data_file.stat().st_size/1e6:.1f} MB")

print("Loading parquet for integrity check...")
_df_check = pd.read_parquet(data_file)
n_rows, n_cols = _df_check.shape
print(f"Shape: {n_rows} x {n_cols}")

REQUIRED_COLS = ["gm_well_id", "latitude", "longitude", "PFOA_ngL"]
missing_cols = [c for c in REQUIRED_COLS if c not in _df_check.columns]
if missing_cols:
    raise ValueError(
        f"INTEGRITY FAIL: columns {missing_cols} missing from the parquet.\n"
        f"File at {data_file} does not match the expected CA-PFAS-ASGWS dataset.\n"
        f"Check DATA_PATH={DATA_PATH!r} or update REPO_URL/GIT_REF."
    )
if n_cols != 201:
    print(f"[WARNING] Expected 201 columns, got {n_cols}. Check dataset version.")
if n_rows != 46338:
    print(f"[WARNING] Expected 46338 rows, got {n_rows}. May be an older snapshot.")
if n_cols == 201 and n_rows == 46338:
    print("[INTEGRITY OK] shape==(46338, 201), required columns present.")
del _df_check

# Override DATA_PARQUET in config so all src/ modules use the same resolved path
import src.config as C
C.DATA_PARQUET = data_file
print(f"src.config.DATA_PARQUET -> {C.DATA_PARQUET}")

Dataset: /home/wiloic/M2/Recherche/tmp/pfas/pfas-gnn/data/CA-PFAS-ASGWS.parquet  size=3.6 MB
Loading parquet for integrity check...
Shape: 46338 x 201
[INTEGRITY OK] shape==(46338, 201), required columns present.
src.config.DATA_PARQUET -> /home/wiloic/M2/Recherche/tmp/pfas/pfas-gnn/data/CA-PFAS-ASGWS.parquet


## Cell 5 — Experiment output directory

Artifacts are written inside the cloned repo workspace (`experiments/<id>/`).
**Warning**: the Colab workspace is ephemeral. See Cell 12 for persistence options
(download archive or git push) to avoid losing results on disconnection.

In [7]:
from pathlib import Path
import time

EXP_ID   = f"baseline_t1{'_smoke' if SMOKE_TEST else ''}"
SAVE_DIR = REPO_LOCAL / "experiments" / EXP_ID
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifacts will be saved to: {SAVE_DIR}")

Artifacts will be saved to: /home/wiloic/M2/Recherche/tmp/pfas/pfas-gnn/experiments/baseline_t1


## Cell 6 — Run baseline T1

**Smoke test** (`SMOKE_TEST=True`): ~500 wells, 3 spatial folds, 3 Optuna trials, CPU < 3 min.

**Full run** (`SMOKE_TEST=False`): all ~11 333 wells, 8 spatial + 8 random folds, 20 Optuna trials.
Estimated duration on CPU: **~45–90 min**.

> Note: scikit-learn and XGBoost use CPUs (`n_jobs=-1`).
> A Colab High-RAM CPU instance may be more efficient than a GPU instance for this baseline.

In [ ]:
import logging, time
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    force=True
)

t_start = time.time()
print(f"{'='*60}")
print(f"T1 BASELINE RUN")
print(f"  target     = {TARGET}")
print(f"  smoke_test = {SMOKE_TEST}")
print(f"  save_dir   = {SAVE_DIR}")
print(f"{'='*60}")

results = run_baselines(
    smoke=SMOKE_TEST,
    target=TARGET,
    run_ablations_flag=True,
    run_shap_flag=True,
    save_dir=SAVE_DIR,
)

elapsed = time.time() - t_start
print(f"\nRun completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")

2026-06-20 10:24:18,192 INFO run_baselines: target=T1a smoke=False outer_k=8 inner_k=4 n_trials=20


T1 BASELINE RUN
  target     = T1a
  smoke_test = False
  save_dir   = /home/wiloic/M2/Recherche/tmp/pfas/pfas-gnn/experiments/baseline_t1


2026-06-20 10:24:18,746 INFO Data: (46338, 193)  wells=11333
2026-06-20 10:24:18,767 INFO Target T1a: prevalence=0.445  n_pos=20633/46338
2026-06-20 10:24:19,410 INFO Spatial blocks: 8  prev [0.21,0.58]
2026-06-20 10:24:19,415 INFO Feature candidates: 96
2026-06-20 10:24:19,415 INFO [LR] outer CV (spatial+random) ...
2026-06-20 10:25:09,204 INFO   [LR/spatial] fold 0: AUC=0.521  recall=0.803  τ=0.10
2026-06-20 10:26:02,159 INFO   [LR/spatial] fold 1: AUC=0.492  recall=0.766  τ=0.10
2026-06-20 10:26:43,462 INFO   [LR/spatial] fold 2: AUC=0.534  recall=0.904  τ=0.10
2026-06-20 10:27:38,865 INFO   [LR/spatial] fold 3: AUC=0.437  recall=0.874  τ=0.10
2026-06-20 10:28:30,523 INFO   [LR/spatial] fold 4: AUC=0.663  recall=0.992  τ=0.10
2026-06-20 10:29:24,283 INFO   [LR/spatial] fold 5: AUC=0.579  recall=0.982  τ=0.10
2026-06-20 10:30:17,085 INFO   [LR/spatial] fold 6: AUC=0.664  recall=0.939  τ=0.10
2026-06-20 10:31:12,255 INFO   [LR/spatial] fold 7: AUC=0.590  recall=0.999  τ=0.10
2026-06-2

## Cell 7 — Results: models x 5 metrics (spatial CV + random CV + Delta)

In [ ]:
import pandas as pd
import numpy as np

print("\n" + "="*80)
print(f"MODELS x 5 METRICS — Target={TARGET}  smoke={SMOKE_TEST}")
print("="*80)

rows = []
for mn, res in results["model_results"].items():
    sp, rd, dlt = res["spatial"], res["random"], res["delta"]
    g = lambda d, k: d.get(f"{k}_mean", float("nan"))
    rows.append({
        "model":          mn,
        "AUC-ROC (sp)":  g(sp, "roc_auc"),
        "F1 (sp)":       g(sp, "f1"),
        "Accuracy (sp)": g(sp, "accuracy"),
        "Recall (sp)":   g(sp, "recall"),
        "Precision (sp)": g(sp, "precision"),
        "AUC-ROC (rd)":  g(rd, "roc_auc"),
        "F1 (rd)":       g(rd, "f1"),
        "Delta AUC-ROC": dlt.get("roc_auc", float("nan")),
        "Delta F1":      dlt.get("f1", float("nan")),
        "Global OOF AUC": res.get("global_auc_spatial_oof", float("nan")),
    })

df_res = pd.DataFrame(rows).set_index("model")
pd.options.display.float_format = "{:.3f}".format
print(df_res.to_string())

print("\nInterpretation:")
print("  Delta AUC-ROC = random - spatial (positive -> spatial autocorrelation artefact).")
print("  Spatial CV is the reference; random CV inflates scores when wells are autocorrelated.")

## Cell 8 — Paired comparisons (Nadeau-Bengio + Wilcoxon)

In [ ]:
print("\n" + "="*60)
print("PAIRED COMPARISONS (spatial folds)")
print("="*60)

for pair, comp in results["comparison"].items():
    nb = comp["nadeau_bengio"]
    wc = comp["wilcoxon"]
    a, b = pair.split("_vs_")
    sig_nb = "*" if (not np.isnan(nb["p"]) and nb["p"] < 0.05) else ""
    sig_wc = "*" if (not np.isnan(wc["p"]) and wc["p"] < 0.05) else ""
    print(f"  {a} vs {b}:")
    print(f"    mean DAUC = {nb['mean_diff']:+.4f}")
    print(f"    Nadeau-Bengio corrected t-test: p = {nb['p']:.4f}{sig_nb}")
    print(f"    Wilcoxon signed-rank:           p = {wc['p']:.4f}{sig_wc}")
    print(f"    (noise threshold = {comp.get('noise_threshold_auc', 0.03)})")
    print()

## Cell 9 — Top SHAP / feature importances

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

imp = results["importance"]
if not imp.empty:
    top_n = min(20, len(imp))
    top = imp.head(top_n)
    method = top["method"].iloc[0]
    print(f"\nTop {top_n} features  [{method}]:\n")
    for _, row in top.iterrows():
        bar = "#" * int(row["importance"] / (imp["importance"].max() + 1e-9) * 30)
        print(f"  {row['feature']:<40} {row['importance']:.4f}  {bar}")

    try:
        fig, ax = plt.subplots(figsize=(8, top_n * 0.35 + 1))
        ax.barh(top["feature"][::-1], top["importance"][::-1], color="steelblue")
        ax.set_xlabel(f"Importance ({method})")
        ax.set_title(f"Top {top_n} features — T1 baseline")
        plt.tight_layout()
        fig_path = SAVE_DIR / "feature_importance.png"
        plt.savefig(fig_path, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Figure saved to {fig_path}")
    except Exception as e:
        print(f"[plot skipped: {e}]")
else:
    print("No feature importance computed (SHAP/permutation was disabled or failed).")

## Cell 10 — Ablations (feature configuration)

In [ ]:
abl = results["ablations"]
if abl:
    print("\n" + "="*60)
    print("ABLATIONS — RF, no Optuna (speed), spatial + random CV")
    print("="*60)
    abl_rows = []
    for key, v in abl.items():
        abl_rows.append({
            "config":     key,
            "n_features": v["n_features"],
            "AUC sp":     v["spatial_roc_auc"],
            "AUC rd":     v["random_roc_auc"],
            "Delta AUC":  v["delta_roc_auc"],
            "Recall sp":  v["spatial_recall"],
        })
    df_abl = pd.DataFrame(abl_rows).set_index("config")
    print(df_abl.to_string(float_format="{:.3f}".format))
    print()
    print("Ablation configurations:")
    print("  no_loc_all   — all features except lat/lon (reference)")
    print("  with_loc_all — include lat/lon as features (spatial leakage risk)")
    print("  no_loc_core  — core hydrogeochemical cocontaminants only")
    print("  no_loc_none  — no cocontaminants")
else:
    print("Ablations not run (run_ablations_flag was False).")

## Cell 11 — Spatial artefact analysis

In [ ]:
bp = results["block_prevalence"]
print("\n" + "="*60)
print("SPATIAL BLOCKS — prevalence per block")
print("="*60)
print(f"  {len(bp)} blocks, prevalence range: "
      f"[{bp['prevalence'].min():.3f}, {bp['prevalence'].max():.3f}]")
print(f"  std(prevalence) = {bp['prevalence'].std():.3f}")
print()
print("  Block-level prevalence variation indicates geographic heterogeneity.")
print("  High variation -> spatial CV is essential to avoid overoptimistic scores.")
print(bp.to_string(index=False))

## Cell 12 — Save summary, timing & persistence

**WARNING — Colab workspace is ephemeral**: files in `/content/` are lost when the
runtime disconnects. Run the persistence cell below to either:
- **Option A**: download an archive of `experiments/<id>/` to your local machine.
- **Option B**: commit and push results back to GitHub (requires authentication).

In [ ]:
import json

print(f"\nArtefacts written to: {SAVE_DIR}")
for f in sorted(SAVE_DIR.iterdir()):
    print(f"  {f.name:40s}  {f.stat().st_size/1024:.1f} KB")

# Timing summary + extrapolation
from src.baselines_t1 import SMOKE_N_WELLS, SMOKE_OUTER_K, OPTUNA_TRIALS_SMOKE, OUTER_SPATIAL_K, OPTUNA_TRIALS_FULL
print(f"\nRun duration: {elapsed:.1f}s ({elapsed/60:.2f} min)")
if SMOKE_TEST:
    n_full = 11333
    scale = (n_full / SMOKE_N_WELLS) * (OUTER_SPATIAL_K / SMOKE_OUTER_K) * (OPTUNA_TRIALS_FULL / OPTUNA_TRIALS_SMOKE)
    est_min = elapsed * scale / 60
    print(f"\nEXTRAPOLATED full run: ~{est_min:.0f} min on CPU")
    print(f"  (x{n_full/SMOKE_N_WELLS:.0f} wells, x{OUTER_SPATIAL_K/SMOKE_OUTER_K:.0f} folds, x{OPTUNA_TRIALS_FULL/OPTUNA_TRIALS_SMOKE:.0f} Optuna trials)")
    print(f"  On Colab (n_jobs=-1): expect ~20-45 min.")
    print()
    print("SMOKE TEST PASSED. To run the full pipeline, set SMOKE_TEST=False.")
else:
    print("FULL RUN COMPLETED.")

## Cell 13 — Persist results (REQUIRED to avoid loss on Colab disconnect)

Choose ONE of the two options below. Without this step, all results are lost
when the Colab runtime terminates.

**Option A** (recommended — no auth needed): download a zip archive to your browser.

**Option B** (git push — requires GitHub token or SSH key): commit the results
back to the repository.

In [ ]:
# ============================================================
#  PERSISTENCE — run this cell before the session ends!
# ============================================================

# --- OPTION A: download archive to your local machine ---
# (Works in Colab without any authentication)
if IN_COLAB:
    import shutil
    from pathlib import Path
    archive_name = f"t1_results_{EXP_ID}"
    archive_path = Path("/content") / archive_name
    shutil.make_archive(str(archive_path), "zip", root_dir=str(SAVE_DIR.parent),
                        base_dir=SAVE_DIR.name)
    zip_file = str(archive_path) + ".zip"
    print(f"Archive created: {zip_file}")
    print("Downloading to browser...")
    try:
        from google.colab import files
        files.download(zip_file)
        print("Download started.")
    except Exception as e:
        print(f"[download error: {e}]")
        print(f"Manually download the archive from: {zip_file}")
else:
    print("[Local run] No download needed — results are at:")
    print(f"  {SAVE_DIR}")

print()
print("--- OPTION B: git push (optional) ---")
print("# Uncomment and run in a new cell if you have GitHub credentials:")
print("# import subprocess")
print("# subprocess.run(['git', '-C', str(REPO_LOCAL), 'add', 'experiments/'], check=True)")
print("# subprocess.run(['git', '-C', str(REPO_LOCAL), 'commit', '-m', 'results: T1 baseline'], check=True)")
print("# subprocess.run(['git', '-C', str(REPO_LOCAL), 'push'], check=True)")